In [9]:
import pandas as pd
from sqlalchemy import create_engine

# Load CSVs
fund_master = pd.read_csv("fund_master.csv")
nav_history = pd.read_csv("nav_history.csv")
transactions = pd.read_csv("investor_transactions.csv")
performance = pd.read_csv("scheme_performance.csv")

# Create database
engine = create_engine("sqlite:///bluestock_mf.db")

# Create tables
fund_master.to_sql(
    "dim_fund",
    engine,
    if_exists="replace",
    index=False
)

nav_history.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

transactions.to_sql(
    "fact_transactions",
    engine,
    if_exists="replace",
    index=False
)

performance.to_sql(
    "fact_performance",
    engine,
    if_exists="replace",
    index=False
)

print("Database tables created successfully!")

Database tables created successfully!


In [10]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("bluestock_mf.db")

pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)

,name
0,dim_fund
1,fact_nav
2,fact_transactions
3,fact_performance


In [12]:
# KPI creation
#KPI 1: Total Investment
total_investment = transactions['amount_inr'].sum()

print(f"Total Investment: ₹{total_investment:,.2f}")

Total Investment: ₹3,521,580,430.00


In [13]:
# KPI 2: Total Investors
total_investors = transactions['investor_id'].nunique()

print("Total Investors:", total_investors)

Total Investors: 5000


In [14]:
# KPI 3: Average Transaction
avg_transaction = transactions['amount_inr'].mean()

print(f"Average Transaction: ₹{avg_transaction:,.2f}")

Average Transaction: ₹107,437.32


In [18]:
# KPI 4: SIP Count
sip_count = transactions[
    transactions['transaction_type'] == 'SIP'
].shape[0]

print("SIP Transactions:", sip_count)

SIP Transactions: 19716


In [17]:
transactions['transaction_type'].value_counts()

transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

In [19]:
kpi_summary = pd.DataFrame({
    'Metric': [
        'Total Investment',
        'Total Investors',
        'Average Transaction',
        'SIP Count'
    ],
    'Value': [
        total_investment,
        total_investors,
        avg_transaction,
        sip_count
    ]
})

kpi_summary

,Metric,Value
0,Total Investment,3.521580e+09
1,Total Investors,5.000000e+03
2,Average Transaction,1.074373e+05
3,SIP Count,1.971600e+04


In [24]:

kpi_summary.to_csv(
    r'C:\Users\shub2\OneDrive\Desktop\bluestock_mf_capstone\data/processed/kpi_summary.csv',
    index=False
)

In [25]:
#Advanced SQL Query 1
# Top 10 Fund Houses by Number of Funds
query = """
SELECT
    fund_house,
    COUNT(*) AS total_funds
FROM dim_fund
GROUP BY fund_house
ORDER BY total_funds DESC
LIMIT 10
"""

pd.read_sql(query, conn)

,fund_house,total_funds
0,SBI Mutual Fund,5
1,Nippon India MF,5
2,ICICI Prudential MF,5
3,HDFC Mutual Fund,5
4,Kotak Mahindra MF,4
5,Axis Mutual Fund,4
6,UTI Mutual Fund,3
7,Mirae Asset MF,3
8,DSP Mutual Fund,3
9,Aditya Birla Sun Life MF,3


In [26]:
#Advanced SQL Query 2
# Investment by Age Group
query = """
SELECT
    age_group,
    SUM(amount_inr) total_investment
FROM fact_transactions
GROUP BY age_group
ORDER BY total_investment DESC
"""

pd.read_sql(query, conn)

,age_group,total_investment
0,26-35,1451600218
1,36-45,871647528
2,18-25,531639392
3,46-55,405406469
4,56+,261286823


In [27]:
# Advanced SQL Query 3
# Gender Wise Investment
query = """
SELECT
    gender,
    SUM(amount_inr) total_investment
FROM fact_transactions
GROUP BY gender
"""

pd.read_sql(query, conn)

,gender,total_investment
0,Female,1176346701
1,Male,2345233729


In [28]:
#Advanced SQL Query 4
# Average Investment by Income Group
query = """
SELECT
    age_group,
    AVG(amount_inr) avg_investment
FROM fact_transactions
GROUP BY age_group
"""

pd.read_sql(query, conn)

,age_group,avg_investment
0,18-25,108144.709520
1,26-35,107821.452722
2,36-45,107003.133808
3,46-55,107278.769251
4,56+,105613.105497


In [29]:
# Advanced SQL Query 5
# Top 10 Funds by Sharpe Ratio
query = """
SELECT
    amfi_code,
    sharpe_ratio
FROM fact_performance
ORDER BY sharpe_ratio DESC
LIMIT 10
"""

pd.read_sql(query, conn)

,amfi_code,sharpe_ratio
0,120507,7.68
1,120844,6.18
2,101208,5.14
3,100025,1.84
4,119120,1.52
5,118636,1.33
6,100016,1.06
7,148567,1.06
8,120504,1.03
9,118632,1.00


In [32]:
# Dashboard Dataset
dashboard_df = transactions.merge(
    fund_master,
    on='amfi_code',
    how='left'
)

In [33]:
dashboard_df = dashboard_df.merge(
    performance,
    on='amfi_code',
    how='left'
)

In [34]:
dashboard_df.shape

(32778, 45)

In [36]:
dashboard_df.to_csv(
    r'C:\Users\shub2\OneDrive\Desktop\bluestock_mf_capstone/data/processed/dashboard_dataset.csv',
    index=False
)

In [42]:
insights = [
    "Equity funds dominate investments",
    "Most investors belong to age 26-35",
    "T30 cities contribute majority of investment",
    "SIP is the preferred investment mode"
]

pd.DataFrame(
    insights,
    columns=['Insights']
).to_csv(
    r'C:\Users\shub2\OneDrive\Desktop\bluestock_mf_capstone\data/business_insights.csv',
    index=False
)